In [19]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

def send_telegram_photo(file_path, caption=""):
    token = os.getenv('TELEGRAM_TOKEN')
    chat_id = os.getenv('CHAT_ID')
    url = f"https://api.telegram.org/bot{token}/sendPhoto"
    
    # 파일을 이진 읽기 모드('rb')로 엽니다.
    with open(file_path, 'rb') as photo:
        files = {'photo': photo}
        params = {
            'chat_id': chat_id,
            'caption': caption  # 이미지와 함께 보낼 텍스트 (선택 사항)
        }
        
        response = requests.post(url, params=params, files=files)
        
    return response.json()


# 일주일간 개별 종목 수익률

In [2]:
import FinanceDataReader as fdr
import pandas as pd
from datetime import datetime, timedelta


portfolio = {
    '종목명': ['삼성전자', 'SK하이닉스', '에이피알','삼성전기'],
    '종목코드': ['005930', '000660','278470','009150'], 
}

df = pd.DataFrame(portfolio)

In [3]:
def return_asset(ticker):

    # 오늘 날짜와 어제 날짜를 자동으로 계산
    today = datetime.today()
    yesterday = today - timedelta(days=10)

    # 계산된 날짜를 그대로 넣어줍니다.
    data = fdr.DataReader(ticker, yesterday, today)['Close']

    close_price = data.iloc[-1]

    returns = (close_price/data.iloc[-5]-1)*100

    return returns,close_price

In [4]:
# 1. 함수를 한 번만 실행하여 결과를 리스트(또는 시리즈) 형태로 저장
results = df['종목코드'].apply(return_asset)

# 2. zip(*results)를 사용하여 튜플을 각각의 컬럼으로 분리
df['수익률_1W'], df['종가'] = zip(*results)

# 3. 수익률만 소수점 처리
df['수익률_1W'] = df['수익률_1W'].round(2)

In [5]:
import platform
import matplotlib.pyplot as plt


# 1. 한글 폰트 및 마이너스 기호 깨짐 방지 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
    
# 마이너스 기호 깨짐 방지 (이 부분이 중요합니다)
plt.rcParams['axes.unicode_minus'] = False

# 2. 그래프 크기 설정
plt.figure(figsize=(10, 6))

# 3. 막대그래프 그리기
# 수익률에 따라 색상을 다르게 주면 더 직관적입니다 (양수: 빨강, 음수: 파랑)
colors = ['red' if x >= 0 else 'blue' for x in df['수익률_1W']]

bars = plt.bar(
    df['종목명'], 
    df['수익률_1W'], 
    color=colors,     # 동적 색상 적용
    edgecolor='black', 
    linewidth=1.2
)

# 4. 막대 위/아래에 정확한 수치(%) 표시하기
for bar in bars:
    height = bar.get_height()
    # 양수면 막대 위(bottom), 음수면 막대 아래(top)에 텍스트 배치
    va = 'bottom' if height >= 0 else 'top'
    
    plt.text(
        bar.get_x() + bar.get_width() / 2.0, 
        height, 
        f'{height:.1f}%', 
        ha='center', 
        va=va, 
        fontsize=12,
        fontweight='bold'
    )

# 5. 축 라벨 및 제목 설정
plt.title('일주일 동안의 보유 주식 주가 변동', fontsize=18, fontweight='bold', pad=20)
plt.ylabel('수익률 (%)', fontsize=12)

# 0선(기준선)을 진하게 그어 마이너스와 플러스를 구분합니다.
plt.axhline(0, color='black', linewidth=0.8)

# 6. y축 범위 자동 설정 (가장 중요!)
# 마이너스 최소값과 플러스 최대값을 찾아 여백을 줍니다.
y_min = df['수익률_1W'].min()
y_max = df['수익률_1W'].max()

# 위아래로 20% 정도 여백 확보
plt.ylim(y_min * 1.2 if y_min < 0 else -1, y_max * 1.2 if y_max > 0 else 1)

plt.tight_layout()

# --- [저장 및 전송 로직] ---
# 1. 파일로 저장 (dpi=300으로 설정하면 이미지가 훨씬 선명해집니다)
image_path = 'stock_return.png'
plt.savefig(image_path, dpi=300)

# 2. 메모리 해제 (여러 번 실행할 경우 필수!)
plt.close()

In [6]:
# 3. 텔레그램으로 전송
today = datetime.today()

# 1. 월과 주차 계산
month = today.month
week_num = (today.day - 1) // 7 + 1

# 2. 캡션 생성
caption_text = f"{month}월 {week_num}번째주\n주식 수익률 현황"

In [7]:
# 3. 전송
result = send_telegram_photo(image_path, caption=caption_text)

if result.get('ok'):
    print("성공적으로 그래프를 전송했습니다.")
else:
    print(f"전송 실패: {result}")

성공적으로 그래프를 전송했습니다.


# 포트폴리오 자산 비중

In [25]:
portfolio = {
    '종목명': ['삼성전자', 'SK하이닉스', '에이피알','삼성전기','예수금(현금)'],
    '종목코드': ['005930', '000660', '278470','009150','CASH'], 
    '수량': [31, 6, 14,7,2489734]
}

df = pd.DataFrame(portfolio)

In [26]:
def get_latest_close(ticker):
    # 종목코드가 'CASH'인 경우 종가를 1원으로 고정
    if ticker == 'CASH':
        return 1    
    try:
        price_data = fdr.DataReader(ticker) 
        return price_data['Close'].iloc[-1] # 가장 마지막(최신) 종가 반환
    except Exception as e:
        print(f"{ticker} 데이터 수집 오류: {e}")
        return 0

In [27]:
df['종가'] = df['종목코드'].apply(get_latest_close)

df['평가금액'] = df['종가'] * df['수량']

total_portfolio_value = df['평가금액'].sum() 

df['비중(%)'] = ((df['평가금액'] / total_portfolio_value) * 100).round(2)

In [31]:
import platform


# 1. 한글 폰트 및 마이너스 기호 깨짐 방지 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')

# 3. 파이 차트 그리기
# 비중(%) 데이터와 종목명 라벨을 매칭합니다.
plt.pie(
    df['비중(%)'], 
    labels=df['종목명'], 
    autopct='%1.1f%%',         # 백분율을 소수점 첫째 자리까지 표시
    wedgeprops={'width': 0.7, 'edgecolor': 'w', 'linewidth': 3} # 도넛 모양 스타일 및 테두리
)

# 4. 제목 추가 및 출력
plt.title('포트폴리오 자산 비중', fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()

# --- [저장 및 전송 로직] ---
# 1. 파일로 저장 (dpi=300으로 설정하면 이미지가 훨씬 선명해집니다)
image_path = 'stock_weight.png'
plt.savefig(image_path, dpi=300)

# 2. 메모리 해제 (여러 번 실행할 경우 필수!)
plt.close()

In [32]:
# 2. 캡션 생성
caption_text = f"{month}월 {week_num}번째주\n포트폴리오 자산 비중"

result = send_telegram_photo(image_path, caption=caption_text)

if result.get('ok'):
    print("성공적으로 그래프를 전송했습니다.")
else:
    print(f"전송 실패: {result}")

성공적으로 그래프를 전송했습니다.


# 일주일간 ETF 수익률

In [39]:
portfolio = {
    '종목명': ['TIGER 반도체TOP10', 'KODEX AI전력핵심설비', 'KODEX 증권','TIGER 코리아원자력','SOL 반도체 전공정','SOL 반도체 후공정','TIGER 은행고배당플러스 TOP10'],
    '종목코드': ['396500', '487240', '102970','0091P0','475300','475310','466940'], 
}

df = pd.DataFrame(portfolio)

In [40]:
# 1. 함수를 한 번만 실행하여 결과를 리스트(또는 시리즈) 형태로 저장
results = df['종목코드'].apply(return_asset)

# 2. zip(*results)를 사용하여 튜플을 각각의 컬럼으로 분리
df['수익률_1W'], df['종가'] = zip(*results)

# 3. 수익률만 소수점 처리
df['수익률_1W'] = df['수익률_1W'].round(2)

In [ ]:
# 1. 한글 폰트 및 마이너스 기호 깨짐 방지 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
    
# 마이너스 기호 깨짐 방지 (이 부분이 중요합니다)
plt.rcParams['axes.unicode_minus'] = False

# 2. 그래프 크기 설정
plt.figure(figsize=(10, 6))

# 3. 막대그래프 그리기
# 수익률에 따라 색상을 다르게 주면 더 직관적입니다 (양수: 빨강, 음수: 파랑)
colors = ['red' if x >= 0 else 'blue' for x in df['수익률_1W']]

bars = plt.bar(
    df['종목명'], 
    df['수익률_1W'], 
    color=colors,     # 동적 색상 적용
    edgecolor='black', 
    linewidth=1.2
)

# 4. 막대 위/아래에 정확한 수치(%) 표시하기
for bar in bars:
    height = bar.get_height()
    # 양수면 막대 위(bottom), 음수면 막대 아래(top)에 텍스트 배치
    va = 'bottom' if height >= 0 else 'top'
    
    plt.text(
        bar.get_x() + bar.get_width() / 2.0, 
        height, 
        f'{height:.1f}%', 
        ha='center', 
        va=va, 
        fontsize=12,
        fontweight='bold'
    )

# 5. 축 라벨 및 제목 설정
plt.title('일주일 동안의 보유 주식 주가 변동', fontsize=18, fontweight='bold', pad=20)
plt.ylabel('수익률 (%)', fontsize=12)

# 0선(기준선)을 진하게 그어 마이너스와 플러스를 구분합니다.
plt.axhline(0, color='black', linewidth=0.8)

# 6. y축 범위 자동 설정 (가장 중요!)
# 마이너스 최소값과 플러스 최대값을 찾아 여백을 줍니다.
y_min = df['수익률_1W'].min()
y_max = df['수익률_1W'].max()

# 위아래로 20% 정도 여백 확보
plt.ylim(y_min * 1.2 if y_min < 0 else -1, y_max * 1.2 if y_max > 0 else 1)

plt.tight_layout()

# --- [저장 및 전송 로직] ---
# 1. 파일로 저장 (dpi=300으로 설정하면 이미지가 훨씬 선명해집니다)
image_path = 'stock_return.png'
plt.savefig(image_path, dpi=300)

# 2. 메모리 해제 (여러 번 실행할 경우 필수!)
plt.close()

In [ ]:
# 1. 한글 폰트 및 마이너스 기호 깨짐 방지 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
    
# 마이너스 기호 깨짐 방지 (이 부분이 중요합니다)
plt.rcParams['axes.unicode_minus'] = False

# 2. 그래프 크기 설정
plt.figure(figsize=(10, 6))

# 3. 막대그래프 그리기
# 수익률에 따라 색상을 다르게 주면 더 직관적입니다 (양수: 빨강, 음수: 파랑)
colors = ['red' if x >= 0 else 'blue' for x in df['수익률_1W']]

bars = plt.bar(
    df['종목명'], 
    df['수익률_1W'], 
    color=colors,     # 동적 색상 적용
    edgecolor='black', 
    linewidth=1.2
)

# 4. 막대 위/아래에 정확한 수치(%) 표시하기
for bar in bars:
    height = bar.get_height()
    # 양수면 막대 위(bottom), 음수면 막대 아래(top)에 텍스트 배치
    va = 'bottom' if height >= 0 else 'top'
    
    plt.text(
        bar.get_x() + bar.get_width() / 2.0, 
        height, 
        f'{height:.1f}%', 
        ha='center', 
        va=va, 
        fontsize=12,
        fontweight='bold'
    )

# 5. 축 라벨 및 제목 설정
plt.title('일주일 동안의 보유 주식 주가 변동', fontsize=18, fontweight='bold', pad=20)
plt.ylabel('수익률 (%)', fontsize=12)

# 0선(기준선)을 진하게 그어 마이너스와 플러스를 구분합니다.
plt.axhline(0, color='black', linewidth=0.8)

# 6. y축 범위 자동 설정 (가장 중요!)
# 마이너스 최소값과 플러스 최대값을 찾아 여백을 줍니다.
y_min = df['수익률_1W'].min()
y_max = df['수익률_1W'].max()

# 위아래로 20% 정도 여백 확보
plt.ylim(y_min * 1.2 if y_min < 0 else -1, y_max * 1.2 if y_max > 0 else 1)

plt.tight_layout()

# --- [저장 및 전송 로직] ---
# 1. 파일로 저장 (dpi=300으로 설정하면 이미지가 훨씬 선명해집니다)
image_path = 'stock_return.png'
plt.savefig(image_path, dpi=300)

# 2. 메모리 해제 (여러 번 실행할 경우 필수!)
plt.close()

In [42]:
# 3. 전송
result = send_telegram_photo(image_path, caption=caption_text)

if result.get('ok'):
    print("성공적으로 그래프를 전송했습니다.")
else:
    print(f"전송 실패: {result}")

성공적으로 그래프를 전송했습니다.
